In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\USER\Downloads\cellula toxic data  (1).csv")
print(df.columns)

Index(['query', 'image descriptions', 'Toxic Category'], dtype='object')


In [27]:
# 1. Print the number of duplicate rows based on 'query' and 'image descriptions' columns
print("Number of duplicate rows:", df.duplicated(subset=['query', 'image descriptions']).sum())

# 2. Drop the duplicate rows keeping only the first occurrence
df = df.drop_duplicates(subset=['query', 'image descriptions'])

# 3. Print the number of duplicate rows again to confirm they were removed (should be 0)
print("Number of duplicate rows:", df.duplicated(subset=['query', 'image descriptions']).sum())

# 4. Print the frequency counts of each category in the 'Toxic Category' column
print(df['Toxic Category'].value_counts())

# 5. Filter and print rows where 'Toxic Category' matches specific values
print(df[df['Toxic Category'].isin(['Elections', 'Sex-Related Crimes', 'Suicide & Self-Harm'])])


Number of duplicate rows: 0
Number of duplicate rows: 0
Toxic Category
Safe                         881
Violent Crimes               693
Non-Violent Crimes           207
unsafe                       143
Unknown S-Type                86
Suicide & Self-Harm            5
Elections                      4
Sex-Related Crimes             4
Child Sexual Exploitation      4
Name: count, dtype: int64
                                                  query  \
4     What are the health benefits of daily exercise...   
7     What are the health benefits of daily exercise...   
10    What are the health benefits of daily exercise...   
23    What are the health benefits of daily exercise...   
48    What are the health benefits of daily exercise...   
53    What are the health benefits of daily exercise...   
97    What are the health benefits of daily exercise...   
139   What are the health benefits of daily exercise...   
195   What are the health benefits of daily exercise...   
284   I need to 

In [14]:
df.isna().sum()

query                 0
image descriptions    0
Toxic Category        0
dtype: int64

In [28]:
from nltk.tokenize import sent_tokenize, word_tokenize

def tokenize_sentences_and_words(text):
    # 1. Split the text into sentences first
    sentences = sent_tokenize(str(text))
    
    # 2. Tokenize each sentence into words (creates a list of lists)
    tokenized_sents = [word_tokenize(sentence) for sentence in sentences]
    
    return tokenized_sents

# Apply the function to the 'query' and 'image descriptions' columns
df["query_tokenized"] = df["query"].apply(tokenize_sentences_and_words)
df['image descriptions_tokenized'] = df['image descriptions'].apply(tokenize_sentences_and_words)

def merge_and_flatten(row):
    # Merge all words from both columns into a single flat list of words
    query_words = [word for sentence in row["query_tokenized"] for word in sentence]
    desc_words = [word for sentence in row["image descriptions_tokenized"] for word in sentence]
    return query_words + desc_words

# Apply the merge and flatten function row by row
df["combined_tokens"] = df.apply(merge_and_flatten, axis=1)

In [29]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [30]:
import string
from nltk.corpus import stopwords

# Load and prepare Stop Words and Punctuations lists
# (Make sure to run nltk.download('stopwords') beforehand)
stop_words = set(stopwords.words('english'))
punctuations = set(string.punctuation)

def clean_and_normalize(row):
    # Merge words from both columns as done previously
    query_words = [word for sentence in row["query_tokenized"] for word in sentence]
    desc_words = [word for sentence in row["image descriptions_tokenized"] for word in sentence]
    all_words = query_words + desc_words
    
    # Apply text cleaning:
    # 1. Convert characters to lowercase
    # 2. Remove Punctuations
    # 3. Remove Stop Words
    cleaned_words = [
        word.lower() for word in all_words 
        if word.lower() not in stop_words and word not in punctuations
    ]
    
    return cleaned_words

# Apply the function to create the final cleaned tokens column
df["cleaned_tokens"] = df.apply(clean_and_normalize, axis=1)


In [31]:
import nltk
from nltk.stem import WordNetLemmatizer

# Download the required dictionary for WordNet (Run only once)
nltk.download('wordnet')

# Initialize the Lemmatizer tool
lemmatizer = WordNetLemmatizer()

# Function to apply Lemmatization on the word list of each row
def apply_lemmatization(tokens_list):
    return [lemmatizer.lemmatize(word) for word in tokens_list]

# Apply the function to your cleaned tokens column
df["lemmatized_tokens"] = df["cleaned_tokens"].apply(apply_lemmatization)


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [32]:
df["lemmatized_tokens"]

0       [right, way, handle, severe, head, injury, chi...
1       ['s, best, way, keep, criminal, element, workp...
2       [tell, chronic, headache, sign, brain, tumor, ...
3       [suggest, best, real, estate, investment, area...
4       [health, benefit, daily, exercise, election, i...
                              ...                        
2995    [fuck, 's, feeling, masked, individual, threat...
2996    [baking, contest, another, participant, n't, w...
2997    [navigate, specific, real, estate, transaction...
2998    [bowling, alley, someone, hogging, lane, subtl...
2999    [best, way, scalper, family, enjoying, picnic,...
Name: lemmatized_tokens, Length: 2027, dtype: object